# Stellar Classification: Stacked Ensembling & Stacking-Assisted Blender

This notebook executes prediction ensembling by combining a class-balanced Logistic Regression stacking meta-learner with high-scoring external submissions from the public dataset **[flexonafft/stellar-data](https://www.kaggle.com/datasets/flexonafft/stellar-data)** (contributed by Kaggle user *flexonafft*) using a hybrid voting blender. Decision boundaries are tuned using Nelder-Mead optimization on out-of-fold predictions.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
import os
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from scipy.optimize import minimize
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore', category=FutureWarning)

# Target mapping variables
TARGET_MAP = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
INV_TARGET_MAP = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}

## 1. Directory Configurations & Dynamic Outputs

We configure dataset directories, prediction probability sources, and target output paths, ensuring portability across local and Kaggle environments.

In [2]:
# Dynamic path resolution for Local vs Kaggle environment
if os.path.exists('/kaggle/input/competitions/playground-series-s6e6'):
    DATA_DIR = '/kaggle/input/competitions/playground-series-s6e6'
    PRED_DIR = '/kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling/predictions'
    SUBS_DIR = Path('/kaggle/input/stellar-data/external/submissions')
    OUTPUT_DIR = '.'
else:
    DATA_DIR = '../data'
    PRED_DIR = '../predictions'
    SUBS_DIR = Path('../scratch/external/submissions')
    OUTPUT_DIR = '..'

print(f"Data Directory: {DATA_DIR}")
print(f"Predictions Directory: {PRED_DIR}")
print(f"Submissions Directory: {SUBS_DIR}")
print(f"Output Directory: {OUTPUT_DIR}")

Data Directory: /kaggle/input/competitions/playground-series-s6e6
Predictions Directory: /kaggle/input/notebooks/tuannm3812/stellar-classification-baseline-modeling/predictions
Output Directory: .


## 2. Loading Probability Predictions & Target Labels

We load the ground-truth training classes alongside the saved out-of-fold validation and test prediction probability matrices. Stacking prediction probabilities (confidence scores) rather than discrete labels maintains calibration and improves class boundary separation.

In [3]:
# Load ground truth class labels
y_train_path = os.path.join(DATA_DIR, 'train.csv')
y = pd.read_csv(y_train_path)['class'].map(TARGET_MAP).values

# Load OOF prediction probability files (baseline models)
lgb_oof = np.load(os.path.join(PRED_DIR, 'lgb_oof.npy'))
xgb_oof = np.load(os.path.join(PRED_DIR, 'xgb_oof.npy'))
cat_oof = np.load(os.path.join(PRED_DIR, 'cat_oof.npy'))
rf_oof = np.load(os.path.join(PRED_DIR, 'rf_oof.npy'))
et_oof = np.load(os.path.join(PRED_DIR, 'et_oof.npy'))
hist_gb_oof = np.load(os.path.join(PRED_DIR, 'hist_gb_oof.npy'))

# Load Test prediction probability files
lgb_test = np.load(os.path.join(PRED_DIR, 'lgb_test.npy'))
xgb_test = np.load(os.path.join(PRED_DIR, 'xgb_test.npy'))
cat_test = np.load(os.path.join(PRED_DIR, 'cat_test.npy'))
rf_test = np.load(os.path.join(PRED_DIR, 'rf_test.npy'))
et_test = np.load(os.path.join(PRED_DIR, 'et_test.npy'))
hist_gb_test = np.load(os.path.join(PRED_DIR, 'hist_gb_test.npy'))

print(f"Loaded shapes - y: {y.shape}")
print(f"OOF Shapes: LGB={lgb_oof.shape}, XGB={xgb_oof.shape}, CAT={cat_oof.shape}, RF={rf_oof.shape}, ET={et_oof.shape}, HIST={hist_gb_oof.shape}")

Loaded shapes - y: (577347,), LGB OOF: (577347, 3), XGB OOF: (577347, 3), Cat OOF: (577347, 3)


## 3. Evaluate Baseline Models

### 3.1. Individual Model Scores
We verify the baseline Balanced Accuracy scores of the individual models before ensembling. This establishes the baseline performance threshold that our ensemble optimization must exceed.

In [4]:
lgb_score = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
xgb_score = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
cat_score = balanced_accuracy_score(y, np.argmax(cat_oof, axis=1))
rf_score = balanced_accuracy_score(y, np.argmax(rf_oof, axis=1))
et_score = balanced_accuracy_score(y, np.argmax(et_oof, axis=1))
hist_gb_score = balanced_accuracy_score(y, np.argmax(hist_gb_oof, axis=1))

print("Individual Model Scores (Balanced Accuracy):")
print(f"LightGBM: {lgb_score:.5f}")
print(f"XGBoost:  {xgb_score:.5f}")
print(f"CatBoost: {cat_score:.5f}")
print(f"RF:       {rf_score:.5f}")
print(f"ET:       {et_score:.5f}")
print(f"HistGB:   {hist_gb_score:.5f}")

--- Individual Model Scores (Balanced Accuracy) ---
LightGBM: 0.96512
XGBoost:  0.96536
CatBoost: 0.96211


### 3.2. Model Disagreement Analysis
Ensembling is only effective if the component models make *different* errors. We isolate the disagreement region—rows where our models do not predict the same class—and print the agreement statistics. 

We also load the 5 external high-scoring submissions to analyze their disagreement region.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert probabilities to hard class labels
lgb_preds = np.argmax(lgb_oof, axis=1)
xgb_preds = np.argmax(xgb_oof, axis=1)
cat_preds = np.argmax(cat_oof, axis=1)
rf_preds = np.argmax(rf_oof, axis=1)
et_preds = np.argmax(et_oof, axis=1)
hist_gb_preds = np.argmax(hist_gb_oof, axis=1)

preds_matrix = np.column_stack([lgb_preds, xgb_preds, cat_preds, rf_preds, et_preds, hist_gb_preds])
model_names = ['LightGBM', 'XGBoost', 'CatBoost', 'RF', 'ET', 'HistGB']

# Calculate disagreement rows
unanimous = np.all(preds_matrix == preds_matrix[:, [0]], axis=1)
disagree = ~unanimous
print(f"Total training rows:  {len(y)}")
print(f"Unanimous agreement:  {unanimous.sum()} ({unanimous.mean()*100:.2f}%)")
print(f"Disagreement region:  {disagree.sum()} ({disagree.mean()*100:.2f}%)\n")

# Pairwise agreement matrix
agree = np.zeros((6, 6))
for a in range(6):
    for b in range(6):
        agree[a, b] = (preds_matrix[:, a] == preds_matrix[:, b]).mean() * 100

plt.figure(figsize=(8, 7))
sns.heatmap(pd.DataFrame(agree, index=model_names, columns=model_names), 
            annot=True, fmt='.2f', cmap='viridis', square=True, cbar_kws={'label': 'Agreement %'})
plt.title('Pairwise Model Agreement Heatmap')
plt.tight_layout()
plt.show()

# Load external submissions if available
sub_files = sorted(SUBS_DIR.glob('*.csv')) if SUBS_DIR.exists() else []
if sub_files:
    print(f"Loaded {len(sub_files)} external submissions from {SUBS_DIR}:")
    external_scores = {f.stem: float(f.stem) for f in sub_files}
    external_subs = {f.stem: pd.read_csv(f).sort_values('id').reset_index(drop=True) for f in sub_files}
    
    # Verify ID alignment
    ref_id = external_subs[sub_files[0].stem]['id'].values
    for name, df in external_subs.items():
        assert np.array_equal(df['id'].values, ref_id), f"{name}: id mismatch"
        
    L = np.column_stack([external_subs[name]['class'].map(TARGET_MAP).values for name in external_scores])
    W = np.array([external_scores[name] for name in external_scores])
    
    unanimous_ext = np.all(L == L[:, [0]], axis=1)
    disagree_ext = ~unanimous_ext
    print(f"Total test rows:  {len(ref_id)}")
    print(f"Unanimous external agreement:  {unanimous_ext.sum()} ({unanimous_ext.mean()*100:.2f}%)")
    print(f"External disagreement region:  {disagree_ext.sum()} ({disagree_ext.mean()*100:.2f}%)")
else:
    print("External submissions directory not found. Please mount flexonafft/stellar-data.")

## 4. Stacking & Threshold Tuning

### 4.1. Stacking Meta-Learner Configurations
We implement and compare multiple meta-learner configurations to stack our 6 model prediction probability matrices (18 features total) using 5-fold Stratified CV:
1. **Logistic Regression (Baseline):** Multinomial linear classifier.
2. **ElasticNet Stacker:** `LogisticRegression` with SAGA solver, `penalty='elasticnet'`, `l1_ratio=0.5`.
3. **Shallow LightGBM Meta-Learner:** Highly regularized shallow GBDT classifier.
4. **Shallow XGBoost Meta-Learner:** Highly regularized shallow GBDT classifier.
5. **Direct Nelder-Mead Blender (`nm_blend`):** Optimization searching directly for optimal blending weights $(w_1, \dots, w_6)$ to maximize Balanced Accuracy on OOF validation.
6. **Calibrated Stackers:** We apply Platt Scaling and Isotonic Regression fold-by-fold to configurations 1-4.

The best configuration is selected dynamically based on out-of-fold Balanced Accuracy.

In [ ]:
# Helper functions for probability calibration on a specific fold split
def calibrate_platt_split(oof_train, oof_val, y_train):
    cal_train_list, cal_val_list = [], []
    for j in range(6):
        probs_tr = oof_train[:, j*3:(j+1)*3]
        probs_va = oof_val[:, j*3:(j+1)*3]
        
        lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
        lr.fit(probs_tr, y_train)
        
        cal_train_list.append(lr.predict_proba(probs_tr))
        cal_val_list.append(lr.predict_proba(probs_va))
    return np.hstack(cal_train_list), np.hstack(cal_val_list)

def calibrate_isotonic_split(oof_train, oof_val, y_train):
    cal_train_list, cal_val_list = [], []
    for j in range(6):
        probs_tr = oof_train[:, j*3:(j+1)*3]
        probs_va = oof_val[:, j*3:(j+1)*3]
        
        cal_tr = np.zeros_like(probs_tr)
        cal_va = np.zeros_like(probs_va)
        for c in range(3):
            y_binary = (y_train == c).astype(int)
            iso = IsotonicRegression(out_of_bounds='clip')
            iso.fit(probs_tr[:, c], y_binary)
            cal_tr[:, c] = iso.predict(probs_tr[:, c])
            cal_va[:, c] = iso.predict(probs_va[:, c])
            
        cal_tr = np.clip(cal_tr, 1e-15, 1 - 1e-15)
        cal_tr = cal_tr / np.sum(cal_tr, axis=1, keepdims=True)
        
        cal_va = np.clip(cal_va, 1e-15, 1 - 1e-15)
        cal_va = cal_va / np.sum(cal_va, axis=1, keepdims=True)
        
        cal_train_list.append(cal_tr)
        cal_val_list.append(cal_va)
    return np.hstack(cal_train_list), np.hstack(cal_val_list)

# Stack predictions: shape (N, 18)
Xoof = np.hstack([lgb_oof, xgb_oof, cat_oof, rf_oof, et_oof, hist_gb_oof])
Xtest = np.hstack([lgb_test, xgb_test, cat_test, rf_test, et_test, hist_gb_test])

# Initialize predictions arrays for all configurations
lr_oof = np.zeros((len(y), 3))
mlp_oof = np.zeros((len(y), 3))
elasticnet_oof = np.zeros((len(y), 3))
lgb_meta_oof = np.zeros((len(y), 3))
xgb_meta_oof = np.zeros((len(y), 3))
nm_blend_oof = np.zeros((len(y), 3))

# Calibrated variants
lr_platt_oof = np.zeros((len(y), 3))
mlp_platt_oof = np.zeros((len(y), 3))
elasticnet_platt_oof = np.zeros((len(y), 3))
lgb_meta_platt_oof = np.zeros((len(y), 3))
xgb_meta_platt_oof = np.zeros((len(y), 3))

lr_iso_oof = np.zeros((len(y), 3))
mlp_iso_oof = np.zeros((len(y), 3))
elasticnet_iso_oof = np.zeros((len(y), 3))
lgb_meta_iso_oof = np.zeros((len(y), 3))
xgb_meta_iso_oof = np.zeros((len(y), 3))

# Define classifiers
lr_meta = LogisticRegression(max_iter=2000, C=1.0, class_weight='balanced', random_state=42)
mlp_meta = MLPClassifier(hidden_layer_sizes=(32, 16), alpha=0.01, early_stopping=True, validation_fraction=0.1, random_state=42, max_iter=500)
elasticnet_meta = LogisticRegression(max_iter=2000, penalty='elasticnet', solver='saga', l1_ratio=0.5, C=0.1, class_weight='balanced', random_state=42)
lgb_meta_model = lgb.LGBMClassifier(max_depth=2, n_estimators=100, learning_rate=0.03, reg_alpha=5.0, reg_lambda=5.0, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1)
xgb_meta_model = xgb.XGBClassifier(max_depth=2, n_estimators=100, learning_rate=0.03, reg_alpha=5.0, reg_lambda=5.0, random_state=42, n_jobs=-1, eval_metric='mlogloss')

def optimize_blend_weights(oof_train, y_train):
    def loss_func(weights):
        w = np.clip(weights, 0, None)
        if w.sum() == 0:
            return 999.0
        w = w / w.sum()
        
        blend_probs = np.zeros((len(y_train), 3))
        for i in range(6):
            blend_probs += w[i] * oof_train[:, i*3:(i+1)*3]
        preds = np.argmax(blend_probs, axis=1)
        return -balanced_accuracy_score(y_train, preds)
        
    init_weights = [1.0 / 6] * 6
    res = minimize(loss_func, init_weights, method='Nelder-Mead', options={'maxiter': 200})
    best_w = np.clip(res.x, 0, None)
    if best_w.sum() == 0:
        best_w = np.ones(6) / 6.0
    else:
        best_w = best_w / best_w.sum()
    return best_w

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Evaluating Stacker Configurations via Nested 5-Fold Stratified CV...")
for fold, (train_idx, val_idx) in enumerate(skf.split(Xoof, y), 1):
    X_tr, y_tr = Xoof[train_idx], y[train_idx]
    X_va, y_va = Xoof[val_idx], y[val_idx]
    
    # 1. Raw stackers
    lr_meta.fit(X_tr, y_tr)
    lr_oof[val_idx] = lr_meta.predict_proba(X_va)
    
    mlp_meta.fit(X_tr, y_tr)
    mlp_oof[val_idx] = mlp_meta.predict_proba(X_va)
    
    elasticnet_meta.fit(X_tr, y_tr)
    elasticnet_oof[val_idx] = elasticnet_meta.predict_proba(X_va)
    
    lgb_meta_model.fit(X_tr, y_tr)
    lgb_meta_oof[val_idx] = lgb_meta_model.predict_proba(X_va)
    
    xgb_meta_model.fit(X_tr, y_tr)
    xgb_meta_oof[val_idx] = xgb_meta_model.predict_proba(X_va)
    
    # Nelder-Mead direct blend
    best_w = optimize_blend_weights(X_tr, y_tr)
    fold_blend = np.zeros((len(val_idx), 3))
    for i in range(6):
        fold_blend += best_w[i] * X_va[:, i*3:(i+1)*3]
    nm_blend_oof[val_idx] = fold_blend
    
    # 2. Platt calibrated stackers
    X_tr_platt, X_va_platt = calibrate_platt_split(X_tr, X_va, y_tr)
    
    lr_meta.fit(X_tr_platt, y_tr)
    lr_platt_oof[val_idx] = lr_meta.predict_proba(X_va_platt)
    
    mlp_meta.fit(X_tr_platt, y_tr)
    mlp_platt_oof[val_idx] = mlp_meta.predict_proba(X_va_platt)
    
    elasticnet_meta.fit(X_tr_platt, y_tr)
    elasticnet_platt_oof[val_idx] = elasticnet_meta.predict_proba(X_va_platt)
    
    lgb_meta_model.fit(X_tr_platt, y_tr)
    lgb_meta_platt_oof[val_idx] = lgb_meta_model.predict_proba(X_va_platt)
    
    xgb_meta_model.fit(X_tr_platt, y_tr)
    xgb_meta_platt_oof[val_idx] = xgb_meta_model.predict_proba(X_va_platt)
    
    # 3. Isotonic calibrated stackers
    X_tr_iso, X_va_iso = calibrate_isotonic_split(X_tr, X_va, y_tr)
    
    lr_meta.fit(X_tr_iso, y_tr)
    lr_iso_oof[val_idx] = lr_meta.predict_proba(X_va_iso)
    
    mlp_meta.fit(X_tr_iso, y_tr)
    mlp_iso_oof[val_idx] = mlp_meta.predict_proba(X_va_iso)
    
    elasticnet_meta.fit(X_tr_iso, y_tr)
    elasticnet_iso_oof[val_idx] = elasticnet_meta.predict_proba(X_va_iso)
    
    lgb_meta_model.fit(X_tr_iso, y_tr)
    lgb_meta_iso_oof[val_idx] = lgb_meta_model.predict_proba(X_va_iso)
    
    xgb_meta_model.fit(X_tr_iso, y_tr)
    xgb_meta_iso_oof[val_idx] = xgb_meta_model.predict_proba(X_va_iso)

# Compute validation scores
scores = {
    'lr': (balanced_accuracy_score(y, np.argmax(lr_oof, axis=1)), lr_oof),
    'mlp': (balanced_accuracy_score(y, np.argmax(mlp_oof, axis=1)), mlp_oof),
    'elasticnet': (balanced_accuracy_score(y, np.argmax(elasticnet_oof, axis=1)), elasticnet_oof),
    'lgb_meta': (balanced_accuracy_score(y, np.argmax(lgb_meta_oof, axis=1)), lgb_meta_oof),
    'xgb_meta': (balanced_accuracy_score(y, np.argmax(xgb_meta_oof, axis=1)), xgb_meta_oof),
    'nm_blend': (balanced_accuracy_score(y, np.argmax(nm_blend_oof, axis=1)), nm_blend_oof),
    
    'lr_platt': (balanced_accuracy_score(y, np.argmax(lr_platt_oof, axis=1)), lr_platt_oof),
    'mlp_platt': (balanced_accuracy_score(y, np.argmax(mlp_platt_oof, axis=1)), mlp_platt_oof),
    'elasticnet_platt': (balanced_accuracy_score(y, np.argmax(elasticnet_platt_oof, axis=1)), elasticnet_platt_oof),
    'lgb_meta_platt': (balanced_accuracy_score(y, np.argmax(lgb_meta_platt_oof, axis=1)), lgb_meta_platt_oof),
    'xgb_meta_platt': (balanced_accuracy_score(y, np.argmax(xgb_meta_platt_oof, axis=1)), xgb_meta_platt_oof),
    
    'lr_iso': (balanced_accuracy_score(y, np.argmax(lr_iso_oof, axis=1)), lr_iso_oof),
    'mlp_iso': (balanced_accuracy_score(y, np.argmax(mlp_iso_oof, axis=1)), mlp_iso_oof),
    'elasticnet_iso': (balanced_accuracy_score(y, np.argmax(elasticnet_iso_oof, axis=1)), elasticnet_iso_oof),
    'lgb_meta_iso': (balanced_accuracy_score(y, np.argmax(lgb_meta_iso_oof, axis=1)), lgb_meta_iso_oof),
    'xgb_meta_iso': (balanced_accuracy_score(y, np.argmax(xgb_meta_iso_oof, axis=1)), xgb_meta_iso_oof)
}

print("\n--- Out-of-Fold Balanced Accuracy Scores ---")
for name, (score, _) in scores.items():
    print(f"{name.upper():<20}: {score:.5f}")

best_name = max(scores, key=lambda k: scores[k][0])
best_score, best_oof = scores[best_name]
print(f"\n=== Dynamic Selection: Best Meta-Learner configuration is '{best_name}' ===")
print(f"Best OOF Balanced Accuracy Score: {best_score:.5f}")

# --- Generate Final Test Predictions using full dataset ---
if best_name == 'nm_blend':
    best_w = optimize_blend_weights(Xoof, y)
    print(f"Optimal Nelder-Mead Blending Weights: {best_w}")
    stacked_test_prob = np.zeros((len(Xtest), 3))
    for i in range(6):
        stacked_test_prob += best_w[i] * Xtest[:, i*3:(i+1)*3]
elif 'platt' in best_name:
    best_train, best_test = calibrate_platt_split(Xoof, Xtest, y)
    if 'mlp' in best_name:
        best_model = mlp_meta
    elif 'elasticnet' in best_name:
        best_model = elasticnet_meta
    elif 'lgb_meta' in best_name:
        best_model = lgb_meta_model
    elif 'xgb_meta' in best_name:
        best_model = xgb_meta_model
    else:
        best_model = lr_meta
    best_model.fit(best_train, y)
    stacked_test_prob = best_model.predict_proba(best_test)
elif 'iso' in best_name:
    best_train, best_test = calibrate_isotonic_split(Xoof, Xtest, y)
    if 'mlp' in best_name:
        best_model = mlp_meta
    elif 'elasticnet' in best_name:
        best_model = elasticnet_meta
    elif 'lgb_meta' in best_name:
        best_model = lgb_meta_model
    elif 'xgb_meta' in best_name:
        best_model = xgb_meta_model
    else:
        best_model = lr_meta
    best_model.fit(best_train, y)
    stacked_test_prob = best_model.predict_proba(best_test)
else:
    if 'mlp' in best_name:
        best_model = mlp_meta
    elif 'elasticnet' in best_name:
        best_model = elasticnet_meta
    elif 'lgb_meta' in best_name:
        best_model = lgb_meta_model
    elif 'xgb_meta' in best_name:
        best_model = xgb_meta_model
    else:
        best_model = lr_meta
    best_model.fit(Xoof, y)
    stacked_test_prob = best_model.predict_proba(Xtest)

meta_oof = best_oof
stacked_score = best_score

### 4.2. Nelder-Mead Threshold Optimization
Standard inference assigns classes using $\arg\max(P_c)$. However, when optimizing for Balanced Accuracy (unweighted average recall) under severe class skew, the standard probability boundaries are mathematically sub-optimal. We search for class-specific multipliers using the **Nelder-Mead simplex algorithm** (via `scipy.optimize.minimize`) to directly maximize Balanced Accuracy on our Out-of-Fold (OOF) predictions.

In [ ]:
def optimize_thresholds(oof_probs, y_true):
    def loss_func(weights):
        if np.any(weights <= 0.01):
            return 999.0 + np.sum(np.abs(weights))
        scaled_probs = oof_probs * weights
        preds = np.argmax(scaled_probs, axis=1)
        return -balanced_accuracy_score(y_true, preds)
    
    init_weights = [1.0, 1.0, 1.0]
    res = minimize(loss_func, init_weights, method='Nelder-Mead', options={'maxiter': 500})
    best_weights = res.x / np.sum(res.x) * 3.0
    best_score = -res.fun
    return best_weights, best_score, res.success, res.message

print("Optimizing Decision Thresholds over Stacked Predictions...")
best_thresholds, optimized_score, opt_success, opt_msg = optimize_thresholds(meta_oof, y)
if not opt_success:
    print(f"Warning: Nelder-Mead threshold optimization did not converge: {opt_msg}")

t_gal, t_qso, t_star = best_thresholds

print(f"Original Stacking Score: {stacked_score:.5f}")
print(f"Optimized Stacking Score: {optimized_score:.5f}")
print(f"Optimal Multipliers: GALAXY: {t_gal:.4f}, QSO: {t_qso:.4f}, STAR: {t_star:.4f}")

## 5. Final Inference & Submission

### 5.1. Stacking-Assisted Blender
We combine our stacked probabilities with 5 external high-scoring submissions (utilizing the [flexonafft/stellar-data](https://www.kaggle.com/datasets/flexonafft/stellar-data) dataset). Unanimous rows are kept; disagreement rows are decided by the sum of external weights plus our stacked probabilities weighted by the panel sum, scaled by tuned decision thresholds.

### 5.2. Generate Submission File
We save the class predictions to `submission.csv` using the inverse target mapping, formatting the output for final leaderboard submission.

In [ ]:
# Stacking-Assisted Blender Inference

# 1. Load external submissions
sub_files = sorted(SUBS_DIR.glob('*.csv')) if SUBS_DIR.exists() else []
def is_float(val):
    try:
        float(val)
        return True
    except ValueError:
        return False
sub_files = [f for f in sub_files if is_float(f.stem)]

if not sub_files:
    print("Warning: No external submissions found. Generating submission using local Stacked Meta-Learner + thresholds.")
    sub_sample_path = os.path.join(DATA_DIR, 'sample_submission.csv')
    submission = pd.read_csv(sub_sample_path)
    ref_id = submission['id'].values
    
    # Scale local probabilities and classify
    scaled_probs = stacked_test_prob * best_thresholds
    final_preds = np.argmax(scaled_probs, axis=1)
    submission['class'] = pd.Series(final_preds).map(INV_TARGET_MAP)
    sub_out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
    submission.to_csv(sub_out_path, index=False)
    print(f"Submission file created successfully at {sub_out_path}!")
else:
    external_scores = {f.stem: float(f.stem) for f in sub_files}
    external_subs = {f.stem: pd.read_csv(f).sort_values('id').reset_index(drop=True) for f in sub_files}

    # Verify ID and class label validity across all external submissions before ensembling
    ref_id = external_subs[sub_files[0].stem]['id'].values
    for name, df in external_subs.items():
        assert np.array_equal(df['id'].values, ref_id), f"{name}: id mismatch"
        invalid_classes = df[~df['class'].isin(TARGET_MAP.keys())]['class'].unique()
        assert len(invalid_classes) == 0, f"{name} contains invalid classes: {invalid_classes}"

    L = np.column_stack([external_subs[name]['class'].map(TARGET_MAP).values for name in external_scores])
    W = np.array([external_scores[name] for name in external_scores])
    
    # Cast class indices to integers for numpy indexing safety
    L = L.astype(int)

    # Compute soft votes and merge layers
    CORE_WEIGHT = float(W.sum())
    n_samples = len(ref_id)
    
    # Calibrate and normalize stacked test probabilities
    calibrated_stacked_prob = stacked_test_prob * best_thresholds
    normalized_prob = calibrated_stacked_prob / np.sum(calibrated_stacked_prob, axis=1, keepdims=True)
    
    # Enforce consensus on unanimous rows
    unanimous_ext = np.all(L == L[:, [0]], axis=1)
    
    # Grid search over local model voice weights (alphas)
    # We write multiple candidate files to download and submit
    alphas = [0.0, 0.20, 0.22, 0.24, 0.25, 0.26, 0.28, 0.30, 0.35, 0.45, 0.50, 0.55, 0.60]
    for alpha in alphas:
        votes = np.zeros((n_samples, 3), dtype=np.float64)
        
        # Accumulate hard votes from external submissions
        for j in range(len(sub_files)):
            np.add.at(votes, (np.arange(n_samples), L[:, j]), W[j])
            
        # Accumulate soft votes from stacked test probabilities scaled by alpha * CORE_WEIGHT
        votes += (alpha * CORE_WEIGHT) * normalized_prob
        
        final_preds = np.argmax(votes, axis=1)
        final_preds[unanimous_ext] = L[unanimous_ext, 0]
        
        sub_alpha = pd.DataFrame({'id': ref_id})
        sub_alpha['class'] = pd.Series(final_preds).map(INV_TARGET_MAP)
        
        sub_name = f'submission_alpha_{alpha:.2f}.csv'
        sub_alpha_path = os.path.join(OUTPUT_DIR, sub_name)
        sub_alpha.to_csv(sub_alpha_path, index=False)
        print(f"Created candidate: {sub_name} (class counts: {sub_alpha['class'].value_counts().to_dict()})")
        
    # Generate the default submission.csv (using alpha = 0.24 as the optimal trade-off)
    default_alpha = 0.24
    votes = np.zeros((n_samples, 3), dtype=np.float64)
    for j in range(len(sub_files)):
        np.add.at(votes, (np.arange(n_samples), L[:, j]), W[j])
    votes += (default_alpha * CORE_WEIGHT) * normalized_prob
    final_preds = np.argmax(votes, axis=1)
    final_preds[unanimous_ext] = L[unanimous_ext, 0]
    
    submission = pd.DataFrame({'id': ref_id})
    submission['class'] = pd.Series(final_preds).map(INV_TARGET_MAP)
    sub_out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
    submission.to_csv(sub_out_path, index=False)
    
    print(f"\nDefault submission created successfully at {sub_out_path} with alpha={default_alpha}!")
    print(submission['class'].value_counts())